In [ ]:
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-6-luna"

In [ ]:
agent = client.beta.agents.create(
    model=MODEL,
    instructions="You speak short sentences. Like caveman.",
)

In [ ]:
import base64

data = b"name:Ax School\nCourses:1000\n"

In [ ]:
with client.beta.agents.sessions.create(
    agent_id=agent.id,
    environment={
        "type": "openai_hosted",
        "container_size": "small",
        "packages": {"python": ["PyYAML==6.0.3"]},
        "network": {
            "access": "disabled",
        },
        "setup_commands": [{"command": "mkdir -p outputs"}],
        "files": [
            {
                "type": "inline",
                "path": "/workspace/course.yaml",
                "data": base64.b64encode(data).decode(),
            }
        ],
    },
    input="Tell me the files i have in my /workspace if they are .yaml open them, and print the parsed object. And save it as JSON to /workspace/outputs/course.json then show me the output of that file.",
    stream=True,
) as events:
    for event in events:
        if event.type == "agent.session.created":
            session_id = event.session.id
        elif event.type == "agent.session.turn.output_text.done":
            print(event.text)
        elif (
            event.type == "agent.session.turn.item.done"
            and event.item.type == "command_execution"
        ):
            print("Command: ", event.item.command)
            print("Output: ", event.item.output)
        else:
            print(event.type)

In [ ]:
artifacts = client.beta.agents.sessions.artifacts.list(session_id)

for artifact in artifacts:
    artifact_id = artifact.id
    print(artifact.to_dict())
    content = client.beta.agents.sessions.artifacts.content(
        artifact_id, session_id=session_id
    )
    print(content.content)

In [93]:
session = client.beta.agents.sessions.retrieve(session_id)

environment = client.beta.agents.environments.retrieve(session.environment.id)

upload = client.beta.agents.environments.files.create(
    environment.id,
    type="inline",
    path="/workspace/hello.txt",
    data=base64.b64encode(b"Hello world lol\n").decode(),
)

upload

EnvironmentFile(environment_id='ccarenv_b64_Y2NhcmVudl82YWUwNmQ5NzEwZmM4MTkxOWVhYmEwZjgxMTdkMDhlNg', object='agent.environment.file', path='/workspace/hello.txt', size_bytes=16)

In [94]:
with client.beta.agents.sessions.events.stream(session_id) as events:
    client.beta.agents.sessions.events.create(
        session_id,
        events=[
            {
                "type": "agent.session.input.message",
                "input": [
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "input_text",
                                "text": "Tell me the files in my workspace again",
                            }
                        ],
                    }
                ],
            }
        ],
    )
    for event in events:
        if event.type == "agent.session.created":
            session_id = event.session.id
        elif event.type == "agent.session.turn.output_text.done":
            print(event.text)
        elif (
            event.type == "agent.session.turn.item.done"
            and event.item.type == "command_execution"
        ):
            print("Command: ", event.item.command)
            print("Output: ", event.item.output)
        else:
            print(event.type)

agent.session.environment.ready
agent.session.environment.connected
agent.session.turn.created
agent.session.in_progress
agent.session.turn.in_progress
agent.session.turn.item.added
agent.session.turn.item.added
Command:  /bin/bash -lc "find /workspace -type f -printf '%p\\n' | sort"
Output:  /workspace/.managed-agents-executor.log
/workspace/.managed-agents-executor.pid
/workspace/.managed-agents/setup/started_at
/workspace/.managed-agents/setup/state
/workspace/course.yaml
/workspace/hello.txt
/workspace/outputs/course.json

agent.session.turn.item.added
agent.session.turn.content_part.added
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_tex

KeyboardInterrupt: 